In [1]:
"""
训练dnn的三种模型，FitAll，FitOne，Dynamic
"""
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr
import random
import copy


IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/"

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

In [2]:
class RiceDataset(Dataset):

    def __init__(self, df, feat_cols, target):

        self.X = df[feat_cols].values.astype(np.float32)
        self.loc = df["location_id"].values.astype(np.int64)
        self.year = df["year_id"].values.astype(np.int64)
        self.y = df[target].values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.loc[idx],
            self.year[idx],
            self.y[idx]
        )


class RiceNet(nn.Module):

    def __init__(self, n_feat, n_loc, n_year):

        super().__init__()

        self.loc_emb = nn.Embedding(n_loc, 2)
        self.year_emb = nn.Embedding(n_year, 2)

        self.mlp = nn.Sequential(
            nn.Linear(n_feat + 4,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x,loc,year):

        loc_e = self.loc_emb(loc)
        year_e = self.year_emb(year)

        z = torch.cat([x,loc_e,year_e],dim=1)

        return self.mlp(z).squeeze(1)


def train_model(model,loader,optimizer,device):

    model.train()

    loss_fn = nn.MSELoss()

    for x,loc,year,y in loader:

        x = x.to(device)
        loc = loc.to(device)
        year = year.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred = model(x,loc,year)

        loss = loss_fn(pred,y)

        loss.backward()

        optimizer.step()


def predict(model,loader,device):

    model.eval()

    preds=[]
    trues=[]

    with torch.no_grad():

        for x,loc,year,y in loader:

            x=x.to(device)
            loc=loc.to(device)
            year=year.to(device)

            pred=model(x,loc,year)

            preds.append(pred.cpu().numpy())
            trues.append(y.numpy())

    preds=np.concatenate(preds)
    trues=np.concatenate(trues)

    return preds,trues

In [3]:
seed=120
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device="cuda" if torch.cuda.is_available() else "cpu"

results=[]

for target_col in traits:

    print("Running trait:",target_col)

    train_df = pd.read_table(os.path.join(IN_PATH, f"trainFill-{target_col}.txt"))
    test_df  = pd.read_table(os.path.join(IN_PATH, f"testFill-{target_col}.txt"))

    feature_cols = [c for c in train_df.columns if c.startswith("fea_")]

    all_loc = pd.concat([train_df["location"], test_df["location"]]).unique()
    all_year = pd.concat([train_df["year"], test_df["year"]]).unique()

    loc2id = {v:i for i,v in enumerate(all_loc)}
    year2id = {v:i for i,v in enumerate(all_year)}

    train_df["location_id"] = train_df["location"].map(loc2id)
    test_df["location_id"]  = test_df["location"].map(loc2id)

    train_df["year_id"] = train_df["year"].map(year2id)
    test_df["year_id"]  = test_df["year"].map(year2id)

    pairs = (
        train_df[["year","location"]]
        .drop_duplicates()
        .sort_values(["year","location"])
    )

    pairs = list(pairs.itertuples(index=False, name=None))


    """
    TrainingWhole
    """

    model_whole = RiceNet(len(feature_cols),len(loc2id),len(year2id)).to(device)

    optimizer = torch.optim.Adam(model_whole.parameters(),lr=1e-4)

    dataset_train = RiceDataset(train_df,feature_cols,target_col)
    train_loader = DataLoader(dataset_train,batch_size=100,shuffle=True)

    for epoch in range(100):
        train_model(model_whole,train_loader,optimizer,device)


    """
    TrainingBaseModel
    """

    base_model = RiceNet(len(feature_cols), len(loc2id), len(year2id)).to(device)

    optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-4)

    for epoch in range(100):
        train_model(base_model,train_loader,optimizer,device)



    """
    循环 pairs
    """

    for i,(year,location) in enumerate(pairs):

        test_data = test_df[
            (test_df["year"]==year) &
            (test_df["location"]==location)
        ]

        if len(test_data)==0:
            continue

        dataset_test = RiceDataset(test_data,feature_cols,target_col)
        test_loader = DataLoader(dataset_test,batch_size=100)


        preds,trues = predict(model_whole,test_loader,device)

        whole_r2 = r2_score(trues,preds)
        whole_rmse = np.sqrt(mean_squared_error(trues,preds))
        whole_pcc = pearsonr(trues,preds)[0]


        """
        TrainingOnebyOne
        """

        model_one = RiceNet(len(feature_cols),len(loc2id),len(year2id)).to(device)

        optimizer = torch.optim.Adam(model_one.parameters(),lr=1e-4)

        train_data = train_df[
            (train_df["year"]==year) &
            (train_df["location"]==location)
        ]

        dataset_train = RiceDataset(train_data,feature_cols,target_col)
        train_loader = DataLoader(dataset_train,batch_size=100,shuffle=True)

        for epoch in range(100):
            train_model(model_one,train_loader,optimizer,device)

        preds,trues = predict(model_one,test_loader,device)

        one_r2 = r2_score(trues,preds)
        one_rmse = np.sqrt(mean_squared_error(trues,preds))
        one_pcc = pearsonr(trues,preds)[0]


        """
        TrainingTwice
        """

        model_twice = copy.deepcopy(base_model)

        for p in model_twice.parameters():
            p.requires_grad=False

        for p in model_twice.mlp[-1].parameters():
            p.requires_grad=True

        optimizer = torch.optim.Adam(
            filter(lambda p:p.requires_grad,model_twice.parameters()),
            lr=1e-5
        )

        dataset_train = RiceDataset(train_data,feature_cols,target_col)
        train_loader = DataLoader(dataset_train,batch_size=100,shuffle=True)

        for epoch in range(10):
            train_model(model_twice,train_loader,optimizer,device)

        preds,trues = predict(model_twice,test_loader,device)

        twice_r2 = r2_score(trues,preds)
        twice_rmse = np.sqrt(mean_squared_error(trues,preds))
        twice_pcc = pearsonr(trues,preds)[0]


        results.append({

            "Trait":target_col,
            "Round":i+1,
            "year":year,
            "location":location,
            "test_data":len(test_data),

            "TraingWhole_R2":whole_r2,
            "TraingWhole_RMSE":whole_rmse,
            "TraingWhole_PCC":whole_pcc,

            "TraingOnebyOne_R2":one_r2,
            "TraingOnebyOne_RMSE":one_rmse,
            "TraingOnebyOne_PCC":one_pcc,

            "TraingTwice_R2":twice_r2,
            "TraingTwice_RMSE":twice_rmse,
            "TraingTwice_PCC":twice_pcc
        })


result_df = pd.DataFrame(results)

print(result_df)

result_df.to_csv(os.path.join(IN_PATH, "pytorch_compare_results.csv"), index=False)

Running trait: Yield_per_plant
Running trait: 1000-grain_weight
Running trait: Grain_length_to_width_ratio
Running trait: Heading_date
Running trait: Plant_height
Running trait: Grain_per_panicle
Running trait: Seed_set_rate
Running trait: Tiller_number
               Trait  Round    year     location  test_data  TraingWhole_R2  \
0    Yield_per_plant      1  2022.0  Gongzhuling         96        0.355242   
1    Yield_per_plant      2  2022.0      Kunming        274        0.708142   
2    Yield_per_plant      3  2022.0      Nanning        353        0.720533   
3    Yield_per_plant      4  2022.0      Tanghai        290        0.555552   
4    Yield_per_plant      5  2023.0  Gongzhuling        195        0.368665   
..               ...    ...     ...          ...        ...             ...   
99     Tiller_number      9  2023.0        Hefei        360       -0.070090   
100    Tiller_number     10  2023.0      Kunming        315        0.259056   
101    Tiller_number     11  2023.0

In [4]:
result_df

,Trait,Round,year,location,test_data,TraingWhole_R2,TraingWhole_RMSE,TraingWhole_PCC,TraingOnebyOne_R2,TraingOnebyOne_RMSE,TraingOnebyOne_PCC,TraingTwice_R2,TraingTwice_RMSE,TraingTwice_PCC
0,Yield_per_plant,1,2022.0,Gongzhuling,96,0.355242,3.899533,0.655183,0.181594,4.393379,0.481846,0.316677,4.014462,0.626969
1,Yield_per_plant,2,2022.0,Kunming,274,0.708142,2.812920,0.857658,0.757067,2.566343,0.870227,0.707136,2.817763,0.859839
2,Yield_per_plant,3,2022.0,Nanning,353,0.720533,3.620488,0.851608,0.719511,3.627100,0.851314,0.735708,3.520815,0.862093
3,Yield_per_plant,4,2022.0,Tanghai,290,0.555552,5.780755,0.747750,0.540936,5.875034,0.736608,0.565423,5.716197,0.753460
4,Yield_per_plant,5,2023.0,Gongzhuling,195,0.368665,3.934956,0.612854,0.204088,4.418169,0.481479,0.405022,3.819975,0.642485
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,Tiller_number,9,2023.0,Hefei,360,-0.070090,2.705931,0.273293,0.176211,2.374184,0.435709,-0.079670,2.718017,0.256944
100,Tiller_number,10,2023.0,Kunming,315,0.259056,1.734044,0.597607,0.294152,1.692478,0.566035,0.339537,1.637162,0.631329
101,Tiller_number,11,2023.0,Nanning,360,0.459697,1.094249,0.715147,0.511329,1.040653,0.715403,0.543737,1.005553,0.763037
102,Tiller_number,12,2023.0,Tanghai,354,0.224052,1.837459,0.543553,0.150219,1.922893,0.477358,0.287877,1.760269,0.575576
